# Presight Warehouse — Interactive SQL Explorer

Runs SQL directly against `outputs/presight_warehouse.duckdb` and renders results as tables, right here in VS Code alongside the rest of the Python solution. Uses the same `duckdb` Python package that built the warehouse — no separate driver/extension involved.

Kernel: any **Python 3** kernel with duckdb/pandas — select it top-right if VS Code opens with a different one.

Edit the query in the last cell and re-run (Shift+Enter) to explore.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

con = duckdb.connect("../../../../../outputs/presight_warehouse.duckdb", read_only=True)
con.sql("SHOW TABLES").df()

,name
0,bridge_employee_project
1,dim_date
2,dim_employee
3,dim_project
4,dim_vendor
5,fact_transactions
6,stg_employees
7,stg_salary_history
8,stg_salary_history_dedup


In [2]:
con.sql("SELECT * FROM dim_project LIMIT 10").df()

,project_key,project_id,project_name,department,status,status_category,start_date,end_date,budget,actual_cost,budget_variance,is_over_budget,duration_days,budget_utilisation_pct,risk_level,project_manager_id,priority,region
0,1,PRJ0001,MLOps Platform,Sustainability,Completed,Closed,2023-06-13,2023-12-01,1200000.0,907809.0,-292191.0,False,171,75.650750,Low,EMP0126,Medium,Dubai
1,2,PRJ0002,Tax Automation Platform - Q2,Marketing,In Progress,Active,2025-02-28,NaT,800000.0,795986.0,-4014.0,False,<NA>,99.498250,Medium,EMP0045,Medium,Abu Dhabi
2,3,PRJ0003,ERP Consolidation Phase 2,Legal,Completed,Closed,2023-11-21,2024-12-16,80000.0,56981.0,-23019.0,False,391,71.226250,Low,EMP0405,Medium,Dubai
3,4,PRJ0004,Employee Wellness App - Q1,Marketing,Completed,Closed,2024-06-09,2025-08-17,800000.0,707314.0,-92686.0,False,434,88.414250,Low,EMP0270,Medium,Abu Dhabi
4,5,PRJ0005,Mobile Workforce App - Q4,Finance,In Progress,Active,2023-04-25,NaT,2000000.0,1329426.0,-670574.0,False,<NA>,66.471300,Medium,EMP0478,High,Ras Al Khaimah
5,6,PRJ0006,Risk Assessment Framework Phase 2,Sustainability,In Progress,Active,2023-09-08,NaT,2000000.0,1438154.0,-561846.0,False,<NA>,71.907700,Medium,EMP0656,High,Abu Dhabi
6,7,PRJ0007,Mobile Workforce App,Finance,Completed,Closed,2022-10-17,2024-01-30,500000.0,437614.0,-62386.0,False,470,87.522800,Low,EMP0188,Medium,Dubai
7,8,PRJ0008,Regulatory Compliance Tool Phase 3,Finance,In Progress,Active,2023-04-29,NaT,200000.0,172385.0,-27615.0,False,<NA>,86.192500,Low,EMP0852,Medium,Dubai
8,9,PRJ0009,IoT Sensor Network,Digital,In Progress,Active,2022-07-24,NaT,1200000.0,1115876.0,-84124.0,False,<NA>,92.989667,Medium,EMP0443,High,Sharjah
9,10,PRJ0010,Energy Optimisation Platform - Q4,Procurement,In Progress,Active,2022-07-16,NaT,800000.0,541681.0,-258319.0,False,<NA>,67.710125,High,EMP0909,Critical,Abu Dhabi


In [3]:
# Q1 — department budget performance
con.sql("""
    SELECT department, SUM(budget) AS total_budget, SUM(actual_cost) AS total_actual_cost,
           ROUND(100.0 * SUM(actual_cost) / NULLIF(SUM(budget), 0), 2) AS spend_percentage
    FROM dim_project
    GROUP BY department
    ORDER BY spend_percentage DESC
""").df()

,department,total_budget,total_actual_cost,spend_percentage
0,Legal,26490000.0,23959404.0,90.45
1,Procurement,22460000.0,20048714.0,89.26
2,Finance,31070000.0,27581728.0,88.77
3,HR,25410000.0,22481570.0,88.48
4,Marketing,20520000.0,17660106.0,86.06
5,Digital,26070000.0,21973878.0,84.29
6,Data Science,16530000.0,13928128.0,84.26
7,Sales,22180000.0,18532534.0,83.56
8,Engineering,18270000.0,15213837.0,83.27
9,IT,19040000.0,15795593.0,82.96


In [4]:
# Scratch cell — edit this query freely
con.sql("SELECT * FROM dim_employee LIMIT 20").df()

,employee_key,employee_id,full_name,email,department,role,level,salary,hire_date,manager_id,region,status,years_experience,valid_from,valid_to,is_current,change_reason
0,1,EMP0001,Anjali Desai,anjali.desai@presight.ai,Operations,Senior Data Engineer,Junior,9882.0,2019-08-06,EMP0000,Abu Dhabi,Active,6.0,2019-08-06,2021-08-21,False,Initial appointment
1,2,EMP0001,Anjali Desai,anjali.desai@presight.ai,Operations,Senior Data Engineer,Junior,11383.0,2019-08-06,EMP0000,Abu Dhabi,Active,6.0,2021-08-22,2022-10-08,False,Market correction adjustment
2,3,EMP0001,Anjali Desai,anjali.desai@presight.ai,Operations,Senior Data Engineer,Mid,17839.0,2019-08-06,EMP0000,Abu Dhabi,Active,6.0,2022-10-09,9999-12-31,True,NaN
3,4,EMP0002,Mohammed Mansour,mohammed.mansour@presight.ai,Operations,Senior Data Engineer,Senior,33928.0,2014-08-20,EMP0000,Dubai,Active,11.0,2014-08-20,9999-12-31,True,NaN
4,5,EMP0003,Adam Rahman,adam.rahman@presight.ai,Legal,Compliance Analyst,Junior,12743.0,2021-02-17,EMP0000,Abu Dhabi,Active,4.0,2021-02-17,2023-01-20,False,Initial appointment
5,6,EMP0003,Adam Rahman,adam.rahman@presight.ai,Legal,Compliance Analyst,Junior,14807.0,2021-02-17,EMP0000,Abu Dhabi,Active,4.0,2023-01-21,2023-01-31,False,Promoted following review
6,7,EMP0003,Adam Rahman,adam.rahman@presight.ai,Legal,Compliance Analyst,Mid,18763.0,2021-02-17,EMP0000,Abu Dhabi,Active,4.0,2023-02-01,9999-12-31,True,NaN
7,8,EMP0004,Fatima Al Suwaidi,fatima.al.suwaidi@presight.ai,Procurement,Procurement Analyst,Junior,14834.0,2024-05-30,EMP0000,Dubai,Active,2.0,2024-05-30,2025-01-09,False,Initial appointment
8,9,EMP0004,Fatima Al Suwaidi,fatima.al.suwaidi@presight.ai,Procurement,Procurement Analyst,Junior,17737.0,2024-05-30,EMP0000,Dubai,Active,2.0,2025-01-10,2025-01-23,False,Cost of living adjustment
9,10,EMP0004,Fatima Al Suwaidi,fatima.al.suwaidi@presight.ai,Procurement,Procurement Analyst,Junior,18854.0,2024-05-30,EMP0000,Dubai,Active,2.0,2025-01-24,2025-06-28,False,Retention adjustment
